# Comparing concentration across public industry portfolios

## A Fixed Public Design Sample

For industry $i$, let $s_i$ be the sample standard deviation of its daily percentage returns in the fixed design period. Two descriptive allocation rules are

$$
w_i^{\mathrm{IVol}}=\frac{s_i^{-1}}{\sum_{j=1}^{n}s_j^{-1}},
\qquad
w_i^{\mathrm{IVar}}=\frac{s_i^{-2}}{\sum_{j=1}^{n}s_j^{-2}}.
$$

Here, $n=16$ is the number of retained public industry portfolios, $w_i^{\mathrm{IVol}}$ is the inverse volatility weight, and $w_i^{\mathrm{IVar}}$ is the inverse variance weight. Both use only observations through 2023. They are descriptive examples of concentration and are not claims about investment performance.

In [1]:
from io import StringIO
from pathlib import Path
from zipfile import ZipFile
import hashlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.optimization import birkhoff_von_neumann_decomposition, is_doubly_stochastic, is_majorized, majorization_matrix

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "french_49_industries")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "24_portfolio_majorization"
TABLES = BOOK_ROOT / "outputs" / "tables" / "24_portfolio_majorization"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "24_portfolio_majorization"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)


def ordered(values):
    return np.sort(np.asarray(values, dtype=float))[::-1]


def partial_sums(values):
    return np.cumsum(ordered(values))


def concentration_record(name, values):
    weights = np.asarray(values, dtype=float)
    positive = weights[weights > 0]
    squared = float(np.sum(weights ** 2))
    entropy = float(-np.sum(positive * np.log(positive)))
    return {
        "Vector": name,
        "Largest component": float(np.max(weights)),
        "Squared sum": squared,
        "Fourth power sum": float(np.sum(weights ** 4)),
        "Effective count": float(1 / squared),
        "Entropy effective count": float(np.exp(entropy)),
    }


def load_public_returns():
    expected_hash = "8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71A7C1CD20BAB83040DE"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    with ZipFile(DATA_PATH) as archive:
        member = archive.namelist()[0]
        lines = archive.read(member).decode("latin1").splitlines()
    title_index = next(index for index, line in enumerate(lines) if "Average Value Weighted Returns -- Daily" in line)
    header_index = title_index + 1
    end_index = next(index for index in range(header_index + 1, len(lines)) if not lines[index].strip())
    frame = pd.read_csv(StringIO("\n".join(lines[header_index:end_index])))
    frame = frame.rename(columns={frame.columns[0]: "Date"})
    frame["Date"] = pd.to_datetime(frame["Date"].astype(str), format="%Y%m%d")
    industries = list(frame.columns[1:17])
    frame[industries] = frame[industries].apply(pd.to_numeric, errors="coerce")
    frame[industries] = frame[industries].mask(frame[industries] <= -99)
    frame = frame.loc[(frame["Date"] >= "2019-01-01") & (frame["Date"] <= "2025-12-31"), ["Date", *industries]].dropna().reset_index(drop=True)
    assert len(industries) == 16
    assert len(frame) == 1760
    return frame, industries

public, industries = load_public_returns()
design = public[public["Date"] <= pd.Timestamp("2023-12-31")].copy()
later = public[public["Date"] >= pd.Timestamp("2024-01-01")].copy()
assert len(design) == 1258
assert len(later) == 502

provenance = pd.DataFrame([
    {"Item": "Public source", "Value": "Kenneth R. French Data Library, 49 Industry Portfolios Daily"},
    {"Item": "Stored archive", "Value": DATA_PATH.name},
    {"Item": "SHA256", "Value": sha256(DATA_PATH)},
    {"Item": "Return type", "Value": "Value-weighted daily percentage returns"},
    {"Item": "Industries retained", "Value": len(industries)},
    {"Item": "Download during execution", "Value": "None"},
    {"Item": "Simulated observations", "Value": "None"},
])
save_table(provenance, "public_data_provenance", "Public data provenance for the Chapter 25 concentration study.", "tab:chapter25-data-provenance", fit=True)

periods = pd.DataFrame([
    {"Sample": "Design", "First date": design["Date"].min(), "Last date": design["Date"].max(), "Daily observations": len(design), "Use": "Estimate descriptive weights"},
    {"Sample": "Later observations", "First date": later["Date"].min(), "Last date": later["Date"].max(), "Daily observations": len(later), "Use": "Count and provenance only"},
    {"Sample": "Complete fixed window", "First date": public["Date"].min(), "Last date": public["Date"].max(), "Daily observations": len(public), "Use": "Public data boundary"},
])
save_table(periods, "public_sample_periods", "Chronological boundaries of the public data study.", "tab:chapter25-public-samples", fit=True)

standard_deviation = design[industries].std(ddof=1)
variance = design[industries].var(ddof=1)
equal = pd.Series(np.full(len(industries), 1 / len(industries)), index=industries)
inverse_volatility = (1 / standard_deviation) / (1 / standard_deviation).sum()
inverse_variance = (1 / variance) / (1 / variance).sum()

industry_summary = pd.DataFrame({
    "Industry": industries,
    "Mean daily return (%)": design[industries].mean().to_numpy(),
    "Daily volatility (%)": standard_deviation.to_numpy(),
    "Smallest daily return (%)": design[industries].min().to_numpy(),
    "Largest daily return (%)": design[industries].max().to_numpy(),
})
save_table(industry_summary, "industry_design_summary", "Design sample summaries for the sixteen public industries.", "tab:chapter25-industry-summary", fit=True)
industry_summary.to_csv(FIGURE_DATA / "industry_volatility.csv", index=False)

weights = pd.DataFrame({
    "Industry": industries,
    "Equal weight (%)": 100 * equal.to_numpy(),
    "Inverse volatility weight (%)": 100 * inverse_volatility.to_numpy(),
    "Inverse variance weight (%)": 100 * inverse_variance.to_numpy(),
})
save_table(weights, "public_portfolio_weights", "Descriptive portfolio weights estimated with data through 2023.", "tab:chapter25-public-weights", fit=True)
weights.to_csv(FIGURE_DATA / "portfolio_weights.csv", index=False)

figure, axis = plt.subplots(figsize=(11.0, 5.6), constrained_layout=True)
ranked = industry_summary.sort_values("Daily volatility (%)")
axis.bar(ranked["Industry"], ranked["Daily volatility (%)"], color=PALETTE["cyan"])
axis.set_xlabel("Industry")
axis.set_ylabel("Daily volatility (%)")
axis.set_title("Design Sample Volatility Across Public Industries")
axis.tick_params(axis="x", rotation=55)
finish_axis(axis)
save_figure(figure, "industry_volatility")
plt.close(figure)

figure, axis = plt.subplots(figsize=(11.2, 5.8), constrained_layout=True)
positions = np.arange(len(industries))
axis.plot(positions, 100 * equal.to_numpy(), marker="o", color=PALETTE["green"], linewidth=1.8, label="Equal")
axis.plot(positions, 100 * inverse_volatility.to_numpy(), marker="s", color=PALETTE["amber"], linewidth=1.8, label="Inverse volatility")
axis.plot(positions, 100 * inverse_variance.to_numpy(), marker="^", color=PALETTE["rose"], linewidth=1.8, label="Inverse variance")
axis.set_xticks(positions, labels=industries, rotation=55)
axis.set_xlabel("Industry")
axis.set_ylabel("Weight (%)")
axis.set_title("Three Descriptive Weight Vectors")
axis.legend()
finish_axis(axis)
save_figure(figure, "portfolio_weights")
plt.close(figure)

,Item,Value
0,Public source,"Kenneth R. French Data Library, 49 Industry Po..."
1,Stored archive,49_Industry_Portfolios_daily_CSV.zip
2,SHA256,8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71...
3,Return type,Value-weighted daily percentage returns
4,Industries retained,16
5,Download during execution,None
6,Simulated observations,None


,Sample,First date,Last date,Daily observations,Use
0,Design,2019-01-02,2023-12-29,"1,258",Estimate descriptive weights
1,Later observations,2024-01-02,2025-12-31,502,Count and provenance only
2,Complete fixed window,2019-01-02,2025-12-31,"1,760",Public data boundary


,Industry,Mean daily return (%),Daily volatility (%),Smallest daily return (%),Largest daily return (%)
0,Agric,0.07279,1.910473,-14.49,10.73
1,Food,0.036812,1.099936,-8.74,9.07
2,Soda,0.047949,1.305024,-9.27,6.54
3,Beer,0.048426,1.288883,-11.28,11.46
4,Smoke,0.046232,1.468137,-9.72,10.03
5,Toys,0.031232,2.395869,-15.81,14.15
6,Fun,0.05124,2.080268,-15.29,9.77
7,Books,0.056383,1.674106,-12.72,8.52
8,Hshld,0.044547,1.226392,-8.82,10.28
9,Clths,0.05368,1.932903,-13.42,13.96


,Industry,Equal weight (%),Inverse volatility weight (%),Inverse variance weight (%)
0,Agric,6.25,5.112839,3.969461
1,Food,6.25,8.880463,11.975083
2,Soda,6.25,7.484875,8.506999
3,Beer,6.25,7.578609,8.721402
4,Smoke,6.25,6.65329,6.721717
5,Toys,6.25,4.076994,2.523989
6,Fun,6.25,4.69552,3.347917
7,Books,6.25,5.83472,5.169485
8,Hshld,6.25,7.964779,9.632849
9,Clths,6.25,5.05351,3.877872


<Figure size 1100x560 with 1 Axes>

<Figure size 1120x580 with 1 Axes>

## Concentration Order in the Public Example

The ordered cumulative share of the largest $k$ weights is

$$
S_k(w)=\sum_{i=1}^{k}w_{[i]},
\qquad k=1,\ldots,n.
$$

Here, $w_{[i]}$ is the weight ranked $i$ after sorting and $S_k(w)$ is the share assigned to the $k$ largest positions. Comparing the complete sequence of partial sums is stronger than comparing only the largest weight or the squared concentration.

In [2]:
portfolio_vectors = {
    "Equal": equal.to_numpy(),
    "Inverse volatility": inverse_volatility.to_numpy(),
    "Inverse variance": inverse_variance.to_numpy(),
}

public_concentration = pd.DataFrame([concentration_record(name, values) for name, values in portfolio_vectors.items()])
save_table(public_concentration, "public_concentration_statistics", "Concentration summaries for the three descriptive public data portfolios.", "tab:chapter25-public-concentration", fit=True)

public_partial = pd.DataFrame({"Position": np.arange(1, len(industries) + 1), **{name: partial_sums(values) for name, values in portfolio_vectors.items()}})
save_table(public_partial, "public_weight_partial_sums", "Ordered cumulative shares of the three weight vectors.", "tab:chapter25-public-partial-sums", fit=True)
public_partial.to_csv(FIGURE_DATA / "public_weight_partial_sums.csv", index=False)

relations = pd.DataFrame([
    {"Target": "Equal", "Source": "Inverse volatility", "Target majorized by source": is_majorized(equal.to_numpy(), inverse_volatility.to_numpy())},
    {"Target": "Equal", "Source": "Inverse variance", "Target majorized by source": is_majorized(equal.to_numpy(), inverse_variance.to_numpy())},
    {"Target": "Inverse volatility", "Source": "Inverse variance", "Target majorized by source": is_majorized(inverse_volatility.to_numpy(), inverse_variance.to_numpy())},
])
save_table(relations, "public_majorization_relations", "Majorization relations among the public data weight vectors.", "tab:chapter25-public-relations")

figure, axis = plt.subplots(figsize=(9.8, 5.6), constrained_layout=True)
colors = [PALETTE["green"], PALETTE["amber"], PALETTE["rose"]]
for (name, values), color in zip(portfolio_vectors.items(), colors):
    axis.plot(np.arange(0, len(values) + 1), np.r_[0, partial_sums(values)], linewidth=2, color=color, label=name)
axis.set_xlabel("Number of largest weights")
axis.set_ylabel("Cumulative portfolio weight")
axis.set_title("Ordered Cumulative Shares of Public Data Weights")
axis.legend()
finish_axis(axis)
save_figure(figure, "public_weight_partial_sums")
plt.close(figure)

alphas = np.linspace(0, 1, 11)
mixing_rows = []
identity = np.eye(len(industries))
equalizer = np.full((len(industries), len(industries)), 1 / len(industries))
for alpha in alphas:
    matrix = (1 - alpha) * identity + alpha * equalizer
    mixed = matrix @ inverse_variance.to_numpy()
    record = concentration_record(f"alpha={alpha:.1f}", mixed)
    mixing_rows.append({"Mixing fraction": alpha, **{key: value for key, value in record.items() if key != "Vector"}, "Majorized by original": is_majorized(mixed, inverse_variance.to_numpy())})
mixing_path = pd.DataFrame(mixing_rows)
save_table(mixing_path, "public_mixing_path", "Concentration along the equalizing doubly stochastic path.", "tab:chapter25-public-mixing-path", fit=True)
mixing_path.to_csv(FIGURE_DATA / "mixing_concentration_path.csv", index=False)

figure, axis = plt.subplots(figsize=(9.6, 5.3), constrained_layout=True)
axis.plot(mixing_path["Mixing fraction"], mixing_path["Squared sum"], marker="o", color=PALETTE["rose"], linewidth=2, label="Squared concentration")
axis.plot(mixing_path["Mixing fraction"], mixing_path["Largest component"], marker="s", color=PALETTE["amber"], linewidth=2, label="Largest weight")
axis.set_xlabel("Equalizing mixing fraction")
axis.set_ylabel("Concentration summary")
axis.set_title("Equalizing Mixing Reduces Concentration")
axis.legend()
finish_axis(axis)
save_figure(figure, "mixing_concentration_path")
plt.close(figure)

figure, axis = plt.subplots(figsize=(9.6, 5.3), constrained_layout=True)
axis.plot(mixing_path["Mixing fraction"], mixing_path["Effective count"], marker="o", color=PALETTE["cyan"], linewidth=2, label="Squared effective count")
axis.plot(mixing_path["Mixing fraction"], mixing_path["Entropy effective count"], marker="s", color=PALETTE["violet"], linewidth=2, label="Entropy effective count")
axis.set_xlabel("Equalizing mixing fraction")
axis.set_ylabel("Effective number of positions")
axis.set_title("Two Effective Counts Rise Toward Sixteen")
axis.legend()
finish_axis(axis)
save_figure(figure, "mixing_effective_count_path")
plt.close(figure)

,Vector,Largest component,Squared sum,Fourth power sum,Effective count,Entropy effective count
0,Equal,0.0625,0.0625,0.000244,16,16
1,Inverse volatility,0.088805,0.065856,0.000325,15.184737,15.570256
2,Inverse variance,0.119751,0.074939,0.000593,13.344209,14.47579


,Position,Equal,Inverse volatility,Inverse variance
0,1,0.0625,0.088805,0.119751
1,2,0.125,0.170957,0.222234
2,3,0.1875,0.250605,0.318562
3,4,0.25,0.326391,0.405776
4,5,0.3125,0.40124,0.490846
5,6,0.375,0.467773,0.558064
6,7,0.4375,0.532462,0.621606
7,8,0.5,0.596601,0.684073
8,9,0.5625,0.655946,0.737552
9,10,0.625,0.714293,0.789247


,Target,Source,Target majorized by source
0,Equal,Inverse volatility,Pass
1,Equal,Inverse variance,Pass
2,Inverse volatility,Inverse variance,Pass


<Figure size 980x560 with 1 Axes>

,Mixing fraction,Largest component,Squared sum,Fourth power sum,Effective count,Entropy effective count,Majorized by original
0,0,0.119751,0.074939,0.000593,13.344209,14.47579,Pass
1,0.1,0.114026,0.072575,0.000521,13.778757,14.762695,Pass
2,0.2,0.108301,0.070461,0.000458,14.192272,15.019525,Pass
3,0.3,0.102576,0.068595,0.000405,14.578312,15.246704,Pass
4,0.4,0.09685,0.066978,0.00036,14.930277,15.444369,Pass
5,0.5,0.091125,0.06561,0.000323,15.241644,15.612466,Pass
6,0.6,0.0854,0.06449,0.000294,15.506227,15.750813,Pass
7,0.7,0.079675,0.063619,0.000272,15.718451,15.859127,Pass
8,0.8,0.07395,0.062998,0.000256,15.873632,15.937055,Pass
9,0.9,0.068225,0.062624,0.000247,15.96822,15.984174,Pass


<Figure size 960x530 with 1 Axes>

<Figure size 960x530 with 1 Axes>

## What the Concentration Study Establishes

At mixing fraction $a\in[0,1]$, the equalizing matrix is

$$
D(a)=(1-a)I+a\frac{\mathbf{1}\mathbf{1}^{\mathsf T}}{n}.
$$

Here, $I$ is the identity matrix, $\mathbf{1}$ is the vector of ones, $n=16$, and $a$ controls the amount of mixing. The matrix is doubly stochastic for every stated $a$. Applying it to the inverse variance vector traces a checked path toward equal weight. This exercise establishes concentration order and numerical reconstruction only. It does not compare later returns, estimate transaction costs, or recommend a portfolio.

In [3]:
source_public = inverse_variance.to_numpy()
witness_rows = []
for target_name, target_public in [("Inverse volatility", inverse_volatility.to_numpy()), ("Equal", equal.to_numpy())]:
    public_witness = majorization_matrix(target_public, source_public)
    witness_rows.append({
        "Target": target_name,
        "Source": "Inverse variance",
        "Doubly stochastic": is_doubly_stochastic(public_witness),
        "Largest mapping residual": np.max(np.abs(public_witness @ source_public - target_public)),
        "Largest row sum residual": np.max(np.abs(public_witness.sum(axis=1) - 1)),
        "Largest column sum residual": np.max(np.abs(public_witness.sum(axis=0) - 1)),
    })
witness_checks = pd.DataFrame(witness_rows)
save_table(witness_checks, "public_witness_checks", "Checked stochastic witnesses for the public data weight vectors.", "tab:chapter25-public-witness-checks", fit=True)

half_matrix = 0.5 * identity + 0.5 * equalizer
decomposition_weights, decomposition_orders = birkhoff_von_neumann_decomposition(half_matrix)
half_restored = sum(weight * np.eye(len(industries))[order] for weight, order in zip(decomposition_weights, decomposition_orders))
decomposition_check = pd.DataFrame([
    {"Quantity": "Returned permutation terms", "Value": len(decomposition_weights)},
    {"Quantity": "Weight sum", "Value": decomposition_weights.sum()},
    {"Quantity": "Smallest positive weight", "Value": decomposition_weights.min()},
    {"Quantity": "Largest reconstruction residual", "Value": np.max(np.abs(half_restored - half_matrix))},
])
save_table(decomposition_check, "public_equalizer_decomposition", "Birkhoff reconstruction of the halfway equalizing matrix.", "tab:chapter25-public-decomposition")

correlation = design[industries].corr()
correlation.to_csv(FIGURE_DATA / "design_correlation_heatmap.csv")
figure, axis = plt.subplots(figsize=(9.2, 7.8), constrained_layout=True)
image = axis.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(np.arange(len(industries)), labels=industries, rotation=55, ha="right")
axis.set_yticks(np.arange(len(industries)), labels=industries)
axis.set_title("Design Sample Industry Return Correlations")
figure.colorbar(image, ax=axis, label="Correlation")
save_figure(figure, "design_correlation_heatmap")
plt.close(figure)

limits = pd.DataFrame([
    {"Question": "Are the vectors ordered by majorization?", "Answered": True, "Boundary": "Within the fixed design with sixteen industries"},
    {"Question": "Do the witnesses reconstruct the targets?", "Answered": True, "Boundary": "Within absolute numerical tolerance"},
    {"Question": "Which vector has the best later return?", "Answered": False, "Boundary": "No performance comparison is made"},
    {"Question": "Are costs and turnover estimated?", "Answered": False, "Boundary": "No trading model is used"},
    {"Question": "Is any portfolio recommended?", "Answered": False, "Boundary": "The study is descriptive"},
])
save_table(limits, "public_study_boundaries", "Questions answered and not answered by the public data study.", "tab:chapter25-public-boundaries", fit=True)

assert relations["Target majorized by source"].all()
assert witness_checks["Doubly stochastic"].all()
assert witness_checks["Largest mapping residual"].max() <= 1e-10
assert np.max(np.abs(half_restored - half_matrix)) <= 1e-10
assert np.all(np.diff(mixing_path["Squared sum"]) <= 1e-12)
assert np.all(np.diff(mixing_path["Effective count"]) >= -1e-12)
display(limits)

,Target,Source,Doubly stochastic,Largest mapping residual,Largest row sum residual,Largest column sum residual
0,Inverse volatility,Inverse variance,Pass,1.59595e-16,1.11022e-15,2.22045e-15
1,Equal,Inverse variance,Pass,6.93889e-17,2.22045e-16,2.22045e-16


,Quantity,Value
0,Returned permutation terms,16
1,Weight sum,1
2,Smallest positive weight,0.03125
3,Largest reconstruction residual,0


<Figure size 920x780 with 2 Axes>

,Question,Answered,Boundary
0,Are the vectors ordered by majorization?,Pass,Within the fixed design with sixteen industries
1,Do the witnesses reconstruct the targets?,Pass,Within absolute numerical tolerance
2,Which vector has the best later return?,Fail,No performance comparison is made
3,Are costs and turnover estimated?,Fail,No trading model is used
4,Is any portfolio recommended?,Fail,The study is descriptive


,Question,Answered,Boundary
0,Are the vectors ordered by majorization?,True,Within the fixed design with sixteen industries
1,Do the witnesses reconstruct the targets?,True,Within absolute numerical tolerance
2,Which vector has the best later return?,False,No performance comparison is made
3,Are costs and turnover estimated?,False,No trading model is used
4,Is any portfolio recommended?,False,The study is descriptive
